# Week 04 Coding Practice: From Similarity Scores to Retrieval

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/week4_coding_practice.ipynb)

Week 03 focused on how text becomes sparse lexical vectors and dense embeddings. This week asks what happens next:

**query -> representation -> similarity scores -> ranking -> interpretation**

This is a formative student notebook. Run it from top to bottom, make a prediction before inspecting each ranking, and write your own responses at the reflection checkpoints. The five core texts and the guiding questions match the Week 04 discussion, so keep evidence you can use in your post.

## Learning objectives

By the end, you will be able to:

- compare Jaccard overlap, TF-IDF cosine similarity, and dense-vector cosine similarity;
- rank candidate texts against a query;
- explain why representation choice changes retrieval behavior;
- test how query wording affects rank order;
- distinguish similarity from task relevance, factual correctness, and usefulness;
- explain why retrieval quality matters to downstream AI systems such as RAG.

## 0. One-time environment setup

The lexical methods use packages already common in Colab. The dense-vector comparison uses the pinned spaCy `en_core_web_md` model. The next cell installs the approved versions only when they are absent or incompatible. In a new Colab runtime, this one-time setup requires internet access and may take a minute.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

SPACY_VERSION = "3.8.16"
MODEL_NAME = "en_core_web_md"
MODEL_VERSION = "3.8.0"
MODEL_URL = (
    "https://github.com/explosion/spacy-models/releases/download/"
    "en_core_web_md-3.8.0/en_core_web_md-3.8.0-py3-none-any.whl"
)

try:
    installed_spacy = version("spacy")
except PackageNotFoundError:
    installed_spacy = None

try:
    installed_model = version("en-core-web-md")
except PackageNotFoundError:
    installed_model = None

packages = []
if installed_spacy != SPACY_VERSION:
    packages.append(f"spacy=={SPACY_VERSION}")
if installed_model != MODEL_VERSION:
    packages.append(MODEL_URL)

if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
    print("Installed the pinned spaCy environment.")
else:
    print("Pinned spaCy environment is already available.")

In [ ]:
import re

import numpy as np
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 100)

# The pretrained word vectors live in the vocabulary, so the tagging and parsing
# components are unnecessary for this activity. Excluding them speeds up execution.
nlp = spacy.load(
    MODEL_NAME,
    exclude=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer", "ner"],
)
assert nlp.vocab.vectors_length == 300
print(f"Loaded {MODEL_NAME} with {nlp.vocab.vectors_length}-dimensional vectors.")

## 1. Inspect the retrieval task

The first five candidates below are exactly the texts used in the Week 04 discussion. Later, two stress-test candidates will make the retrieval problem harder.

In [ ]:
initial_query = "I need a place to work quietly with Wi-Fi."

core_documents = pd.DataFrame(
    {
        "document_id": ["D1", "D2", "D3", "D4", "D5"],
        "text": [
            "Coffee shop with free Wi-Fi and plenty of tables",
            "Quiet public library with study rooms",
            "Best espresso and pastries downtown",
            "Coworking space with desks and internet",
            "How to troubleshoot a wireless router",
        ],
    }
)

print("Query:", initial_query)
core_documents

### Discussion evidence checkpoint 1: predict before running retrieval

Write brief notes before continuing. These map directly to the Week 04 discussion questions.

1. Which candidates should a human consider the best matches for the user's full need? Why?
2. Which candidates look similar based on shared words alone? Name the shared words.
3. Which candidates might a dense-vector system place close to the query even without many shared words?
4. Which result would you expect to trust most for this task, and what makes it useful rather than merely similar?

**Your prediction:**

- Human top matches:
- Lexical top matches:
- Dense-vector top matches:
- Reason for trusting one result:

## 2. Jaccard similarity: a lexical-overlap baseline

Jaccard similarity compares token sets:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

A score of 1 means identical sets and 0 means no overlap. Jaccard **distance** is `1 - similarity`, so always check which quantity a tool returns. Here we compute similarity directly. The small stop-word list and `Wi-Fi` normalization are visible choices, not universal preprocessing rules.

In [ ]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+")
STOP_WORDS = {"a", "an", "and", "i", "of", "the", "to", "with"}


def lexical_tokens(text):
    normalized = text.lower().replace("wi-fi", "wifi")
    return set(TOKEN_PATTERN.findall(normalized)) - STOP_WORDS


def jaccard_details(left_text, right_text):
    left = lexical_tokens(left_text)
    right = lexical_tokens(right_text)
    intersection = left & right
    union = left | right
    score = len(intersection) / len(union) if union else 1.0
    return intersection, union, score


def rank_scores(items, scores, method):
    ranked = items[["document_id", "text"]].copy()
    ranked["score"] = np.asarray(scores, dtype=float)
    ranked = ranked.sort_values(
        ["score", "document_id"], ascending=[False, True], kind="stable"
    ).reset_index(drop=True)
    ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))
    ranked.insert(1, "method", method)
    return ranked


def jaccard_ranking(query, items):
    scores = [jaccard_details(query, text)[2] for text in items["text"]]
    return rank_scores(items, scores, "Jaccard")

In [ ]:
jaccard_initial = jaccard_ranking(initial_query, core_documents)

selected_details = []
for document_id in ["D1", "D2", "D4", "D5"]:
    text = core_documents.loc[core_documents["document_id"] == document_id, "text"].iloc[0]
    intersection, union, score = jaccard_details(initial_query, text)
    selected_details.append(
        {
            "document_id": document_id,
            "shared_tokens": sorted(intersection),
            "union_size": len(union),
            "jaccard_similarity": score,
        }
    )

print("Selected token-set evidence:")
display(pd.DataFrame(selected_details).round(3))
print("Jaccard ranking:")
jaccard_initial.round(3)

### Pause and interpret

- Where does Jaccard agree with your lexical prediction?
- Why can a relevant candidate receive zero lexical overlap?
- Does `quiet` count as the same token as `quietly` here? What does that reveal about this baseline?

## 3. TF-IDF plus cosine similarity

TF-IDF represents each text in a shared sparse feature space. Cosine similarity compares vector direction rather than raw length. We fit the vectorizer once on the candidate collection, then **transform** each query into that same feature space.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
core_tfidf = tfidf_vectorizer.fit_transform(core_documents["text"])


def tfidf_ranking(query, items=core_documents, document_matrix=core_tfidf):
    query_vector = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_matrix).ravel()
    return rank_scores(items, scores, "TF-IDF cosine")


tfidf_initial = tfidf_ranking(initial_query)
tfidf_initial.round(3)

### Debugging checkpoint: one feature space, not two

Consider this incorrect approach:

```python
query_vector = TfidfVectorizer().fit_transform([initial_query])
cosine_similarity(query_vector, core_tfidf)
```

The query vector and document matrix were fitted with different vocabularies, so their columns do not represent the same features. Predict their shapes, then run the correct check below.

**Explain in your own words:** Why is `transform`, rather than another `fit_transform`, required for a new query?

In [ ]:
query_tfidf = tfidf_vectorizer.transform([initial_query])
print("Document matrix shape:", core_tfidf.shape)
print("Query vector shape:  ", query_tfidf.shape)
assert query_tfidf.shape[1] == core_tfidf.shape[1]

shared_query_terms = sorted(
    set(tfidf_vectorizer.get_feature_names_out()) & lexical_tokens(initial_query)
)
print("Query terms present in the fitted document vocabulary:", shared_query_terms)

## 4. Dense vectors plus cosine similarity

Now only the representation changes. We use the same cosine function on 300-dimensional dense vectors from spaCy. `en_core_web_md` forms a document vector from pretrained lexical vectors; it is a useful lightweight baseline, but it is not a dedicated sentence-transformer model and it is not ground truth about meaning.

A dense-vector result may recognize related words that TF-IDF misses. It may also be misled by broad associations, word averaging, missing context, or details that matter to the user's task.

In [ ]:
core_dense = np.vstack([doc.vector for doc in nlp.pipe(core_documents["text"])])


def dense_ranking(query, items=core_documents, document_matrix=core_dense):
    query_vector = nlp(query).vector.reshape(1, -1)
    scores = cosine_similarity(query_vector, document_matrix).ravel()
    return rank_scores(items, scores, "spaCy dense cosine")


dense_initial = dense_ranking(initial_query)
dense_initial.round(3)

In [ ]:
def comparison_table(*rankings):
    pieces = []
    for ranking in rankings:
        method = ranking["method"].iloc[0]
        piece = ranking[["document_id", "rank", "score"]].rename(
            columns={"rank": f"{method} rank", "score": f"{method} score"}
        )
        pieces.append(piece)

    comparison = core_documents.copy()
    for piece in pieces:
        comparison = comparison.merge(piece, on="document_id")
    return comparison


core_comparison = comparison_table(jaccard_initial, tfidf_initial, dense_initial)
core_comparison.round(3)

### Discussion evidence checkpoint 2: compare the rankings

Use document IDs and scores from the table to record evidence for your discussion post. Do not stop at naming the highest score.

1. **Agreement:** Where do two methods rank a candidate similarly? What evidence might explain the agreement?
2. **Meaningful difference:** Identify one candidate whose lexical and dense-vector ranks differ. Which representation choices may explain the change?
3. **Surprise:** Which result most conflicts with your human prediction? Is it actually wrong, merely ambiguous, or useful for a different task?
4. **Trust:** Which result would you trust for the user's full request? Justify the choice using task constraints, not the score alone.

**Your evidence notes:**

- Agreement:
- Difference and likely cause:
- Surprising result:
- Most useful result and justification:

## 5. Query sensitivity: change the wording and intent

A retrieval result belongs to a particular query, representation, and collection. Choose one provided variant below, predict which documents will move, and then run all three retrieval methods again.

- `study`: keeps the quiet-work intent but replaces several words.
- `coffee`: shifts the intent toward a cafe while retaining Wi-Fi.

In [ ]:
query_options = {
    "study": "quiet study space with internet access",
    "coffee": "coffee and Wi-Fi near downtown",
}

query_choice = "study"  # Change to "coffee" and rerun this cell if you wish.
revised_query = query_options[query_choice]

jaccard_revised = jaccard_ranking(revised_query, core_documents)
tfidf_revised = tfidf_ranking(revised_query)
dense_revised = dense_ranking(revised_query)

print("Initial query:", initial_query)
print("Revised query:", revised_query)
comparison_table(jaccard_revised, tfidf_revised, dense_revised).round(3)

In [ ]:
def rank_movement(initial_ranking, revised_ranking):
    method = initial_ranking["method"].iloc[0]
    initial = initial_ranking[["document_id", "rank"]].rename(
        columns={"rank": "initial_rank"}
    )
    revised = revised_ranking[["document_id", "rank", "score"]].rename(
        columns={"rank": "revised_rank", "score": "revised_score"}
    )
    moved = initial.merge(revised, on="document_id").merge(
        core_documents, on="document_id"
    )
    moved["rank_change"] = moved["initial_rank"] - moved["revised_rank"]
    moved.insert(0, "method", method)
    return moved.sort_values(["revised_rank", "document_id"])


movement = pd.concat(
    [
        rank_movement(jaccard_initial, jaccard_revised),
        rank_movement(tfidf_initial, tfidf_revised),
        rank_movement(dense_initial, dense_revised),
    ],
    ignore_index=True,
)
movement.round(3)

### Discussion evidence checkpoint 3: explain the query change

Positive `rank_change` values mean a document moved closer to rank 1.

- Which document moved most under each representation? Cite the rank change.
- Which movement reflects the revised user's intent?
- Which movement may be an artifact of exact wording or broad vector association?
- Did the revised wording improve retrieval, worsen it, or change the task? Explain the criterion you used.

Save one observation for your initial post. For the peer response, you can compare a classmate's surprising result with yours and propose a different query wording that would test their explanation.

## 6. Stress test: similar wording can still miss the task

The next two candidates add task constraints. One is quiet but prohibits electronics; the other repeats several query words but is a router guide rather than a place. Predict how each method will respond before running the cell.

In [ ]:
stress_documents = pd.DataFrame(
    {
        "document_id": ["D6", "D7"],
        "text": [
            "Silent meditation retreat with a no-electronics policy",
            "Guide to finding quiet Wi-Fi channels for work-from-home routers",
        ],
    }
)
all_documents = pd.concat([core_documents, stress_documents], ignore_index=True)

all_tfidf_vectorizer = TfidfVectorizer(stop_words="english")
all_tfidf = all_tfidf_vectorizer.fit_transform(all_documents["text"])
all_dense = np.vstack([doc.vector for doc in nlp.pipe(all_documents["text"])])

jaccard_stress = jaccard_ranking(initial_query, all_documents)
tfidf_stress = rank_scores(
    all_documents,
    cosine_similarity(
        all_tfidf_vectorizer.transform([initial_query]), all_tfidf
    ).ravel(),
    "TF-IDF cosine",
)
dense_stress = rank_scores(
    all_documents,
    cosine_similarity(nlp(initial_query).vector.reshape(1, -1), all_dense).ravel(),
    "spaCy dense cosine",
)

pd.concat([jaccard_stress, tfidf_stress, dense_stress], ignore_index=True).round(3)

### Discussion evidence checkpoint 4: similarity is not usefulness

Choose one stress-test result and answer all four questions.

1. What representation evidence probably raised or lowered its score?
2. Does the text satisfy the user's request for a **place**, **quiet work**, and **Wi-Fi**? Check each constraint.
3. Is the result useful for this task, useful for a different task, or misleading?
4. What additional information or retrieval rule could help?

This distinction is central to the discussion: a high similarity score is not automatically relevance, factual equivalence, correctness, or usefulness.

## 7. Independent retrieval check

Edit `student_query`, state your expected top result, and rerun the next cell. Your query should describe a plausible need that at least one core candidate could satisfy.

In [ ]:
student_query = "Where can I study in silence?"  # Replace with your own query.

student_results = comparison_table(
    jaccard_ranking(student_query, core_documents),
    tfidf_ranking(student_query),
    dense_ranking(student_query),
)
print("Student query:", student_query)
student_results.round(3)

### Independent interpretation

- Expected top result and reason:
- Observed top result under each method:
- Evidence that supports or changes your prediction:
- One limitation of this comparison:

## 8. Why retrieval quality matters to RAG

A simplified retrieval-augmented generation workflow is:

**user query -> retrieve similar documents -> provide retrieved context to an AI model -> generate a response**

This notebook stops at retrieval; it does not implement RAG. But the ranking still matters. If the system retrieves a router guide or a no-electronics retreat for the original query, a downstream model may produce a fluent answer grounded in context that does not satisfy the user's actual need.

### Final discussion

Use your results to prepare the Week 04 initial post:

1. State your human prediction.
2. Cite one agreement and one meaningful difference between lexical and dense-vector rankings.
3. Identify one surprising result and explain whether it is wrong, ambiguous, or useful for another task.
4. Describe what changed after revising the query and why.
5. Decide which retrieved result you trust most for the task and justify that decision with constraints and evidence.
6. Explain one downstream consequence if an AI assistant receives the wrong 'most similar' document.

For your peer response, use a classmate's result to propose a new query wording, predict whether it will improve or worsen retrieval, and name a context where the same result could be acceptable or problematic.

## 9. Transparent validation checks

These checks confirm that the notebook compared aligned representations and produced complete, ordered rankings. They validate computation, not whether a result is useful for a human task.

In [ ]:
def validate_ranking(ranking, expected_size, lower_bound, upper_bound):
    assert len(ranking) == expected_size
    assert ranking["document_id"].is_unique
    assert ranking["rank"].tolist() == list(range(1, expected_size + 1))
    assert np.isfinite(ranking["score"]).all()
    assert ranking["score"].between(lower_bound - 1e-9, upper_bound + 1e-9).all()
    assert ranking["score"].is_monotonic_decreasing


assert len(core_documents) == 5
assert len(all_documents) == 7
assert core_tfidf.shape[0] == len(core_documents)
assert query_tfidf.shape[1] == core_tfidf.shape[1]
assert core_dense.shape == (len(core_documents), 300)
assert all_dense.shape == (len(all_documents), 300)

for ranking in [jaccard_initial, jaccard_revised, jaccard_stress]:
    validate_ranking(ranking, len(ranking), 0.0, 1.0)
for ranking in [tfidf_initial, tfidf_revised, tfidf_stress]:
    validate_ranking(ranking, len(ranking), 0.0, 1.0)
for ranking in [dense_initial, dense_revised, dense_stress]:
    validate_ranking(ranking, len(ranking), -1.0, 1.0)

assert jaccard_initial.iloc[0]["document_id"] == "D1"
assert tfidf_initial.iloc[0]["document_id"] == "D1"
assert jaccard_stress.loc[jaccard_stress["document_id"] == "D7", "score"].iloc[0] > 0

print("All structural, alignment, score-range, and ranking-order checks passed.")

## Takeaway

Cosine similarity does not determine what a text means by itself. It compares vectors, and the representation determines what those vectors preserve. Retrieval adds another layer of judgment: the highest score is useful only when it serves the user's task. Validate rankings with evidence, constraints, and human interpretation before trusting them downstream.